# Camada Gold: Projeto CineData Analytics

**Objetivo:** modelagem dimensional (**Star Schema**) para o time de BI + respostas às perguntas de negócio.

**Decisões de modelagem**

* **Surrogate Keys** `BIGINT` via `row_number()` sobre a chave natural ordenada → determinísticas e sem colisão (diferente de hash).
* **Grão da fato:** 1 linha por filme **lançado** (`status_filme = 'Lançado'`). Como `silver.tb_financeiro_filmes` e `silver.tb_metricas_engajamento` já têm 1 linha por filme, os `LEFT JOIN` não multiplicam linhas (há um `assert` de grão).
* **Pontes (bridge):** relações N:N filme↔gênero/pessoa/produtora ficam fora da fato, evitando duplicar métricas.
* **`dim_people`** tem grão (nome, tipo): a mesma pessoa pode ser Ator e Diretor (2 linhas, 2 SKs); `tipo_pessoa` ∈ {Ator, Diretor, Roteirista}. Produtoras vão para `dim_companies`.
* **`dim_reviews`:** `qtd_avaliacoes_usuarios` = número de avaliações do filme, `nota_media_usuarios` = média (2 casas) das notas **válidas** (NULLs são ignorados pela média). Avaliações de filmes inexistentes em `dim_movies` (órfãs) não entram, para manter integridade referencial.

In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.text("catalogo", "", "")
CATALOGO = dbutils.widgets.get("catalogo").strip()
if CATALOGO:
    spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")


def gravar_gold(df, tabela):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"gold.{tabela}"))
    print(f"gold.{tabela}: {spark.table('gold.' + tabela).count():,} linhas")


def com_sk(df, coluna_sk, *ordem):
    # Surrogate key BIGINT determinística = row_number() sobre a chave natural ordenada.
    return df.withColumn(coluna_sk, F.row_number().over(Window.orderBy(*ordem)).cast("bigint"))

## Entrega 1 — Dimensões

In [0]:
# dim_movies
s_info = spark.table("silver.tb_info_filmes")
dim_movies = (
    com_sk(s_info, "sk_movie_id", F.col("id_filme").cast("bigint"))
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("id_filme").cast("string"),
        F.col("titulo").cast("string"),
        F.col("data_lancamento").cast("date"),
        F.col("ano_lancamento").cast("int"),
        F.col("duracao_minutos").cast("int"),
        F.col("idioma_original").cast("string"),
        F.col("status_filme").cast("string"),
        F.col("sinopse").cast("string"),
    )
)
gravar_gold(dim_movies, "dim_movies")
dim_movies = spark.table("gold.dim_movies") # relê do Delta (SKs congeladas)

# dim_genres
dim_genres = (
    com_sk(spark.table("silver.tb_generos").select(F.col("genero").alias("nome_genero")).distinct(),
           "sk_genre_id", F.col("nome_genero"))
    .select(F.col("sk_genre_id").cast("bigint"), F.col("nome_genero").cast("string"))
)
gravar_gold(dim_genres, "dim_genres")
dim_genres = spark.table("gold.dim_genres")

# dim_people
s_pe = spark.table("silver.tb_pessoas_empresas")
dim_people = (
    com_sk(
        s_pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
            .select(F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa")).distinct(),
        "sk_person_id", F.col("tipo_pessoa"), F.col("nome_pessoa"))
    .select(F.col("sk_person_id").cast("bigint"), F.col("nome_pessoa").cast("string"), F.col("tipo_pessoa").cast("string"))
)
gravar_gold(dim_people, "dim_people")
dim_people = spark.table("gold.dim_people")

# dim_companies
dim_companies = (
    com_sk(
        s_pe.filter(F.col("tipo_entidade") == "Produtora").select(F.col("nome_entidade").alias("nome_produtora")).distinct(),
        "sk_company_id", F.col("nome_produtora"))
    .select(F.col("sk_company_id").cast("bigint"), F.col("nome_produtora").cast("string"))
)
gravar_gold(dim_companies, "dim_companies")
dim_companies = spark.table("gold.dim_companies")

# dim_reviews
resumo_reviews = (
    spark.table("silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
         F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"))
    .join(dim_movies.select("sk_movie_id", "id_filme"), "id_filme", "inner") # descarta avaliações órfãs
)
dim_reviews = (
    com_sk(resumo_reviews, "sk_review_id", F.col("sk_movie_id"))
    .select(F.col("sk_review_id").cast("bigint"), F.col("sk_movie_id").cast("bigint"),
            "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)
gravar_gold(dim_reviews, "dim_reviews")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.dim_movies: 97,611 linhas
gold.dim_genres: 19 linhas
gold.dim_people: 417,393 linhas
gold.dim_companies: 44,543 linhas
gold.dim_reviews: 27,226 linhas


## Entrega 1 — Tabela Fato

Grão: **1 filme lançado**. Todas as métricas vêm da Silver já deduplicada; `LEFT JOIN` mantém filmes sem dados financeiros/engajamento (métricas `NULL`).

In [0]:
fin = spark.table("silver.tb_financeiro_filmes")
met = spark.table("silver.tb_metricas_engajamento")

fact_movies_performance = (
    dim_movies.filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
    .join(fin, "id_filme", "left")
    .join(met, "id_filme", "left")
    .select(
        F.col("sk_movie_id").cast("bigint"),
        F.col("orcamento_usd").cast("decimal(18,2)"),
        F.col("receita_usd").cast("decimal(18,2)"),
        F.col("lucro_usd").cast("decimal(18,2)"),
        F.col("orcamento_brl").cast("decimal(18,2)"),
        F.col("receita_brl").cast("decimal(18,2)"),
        F.col("lucro_brl").cast("decimal(18,2)"),
        F.col("popularidade").cast("double"),
        F.col("nota_media_tmdb").cast("double"),
        F.col("qtd_votos_tmdb").cast("int"),
        F.col("nota_media_imdb").cast("double"),
        F.col("qtd_votos_imdb").cast("int"),
    )
)
# Garantia de grão: nenhum filme duplicado pelos joins
n_total = fact_movies_performance.count()
n_distintos = fact_movies_performance.select("sk_movie_id").distinct().count()
assert n_total == n_distintos, f"Grão violado na fato: {n_total} linhas vs {n_distintos} filmes"
gravar_gold(fact_movies_performance, "fact_movies_performance")

gold.fact_movies_performance: 96,261 linhas


## Entrega 1 — Tabelas-ponte (bridge)

In [0]:
mv = dim_movies.select("sk_movie_id", "id_filme")

bridge_movie_genre = (
    spark.table("silver.tb_generos")
    .join(mv, "id_filme")
    .join(dim_genres, F.col("genero") == F.col("nome_genero"))
    .select("sk_movie_id", "sk_genre_id").distinct()
)
gravar_gold(bridge_movie_genre, "bridge_movie_genre")

bridge_movie_person = (
    s_pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(mv, "id_filme")
    .join(dim_people, (F.col("nome_entidade") == F.col("nome_pessoa")) & (F.col("tipo_entidade") == F.col("tipo_pessoa")))
    .select("sk_movie_id", "sk_person_id").distinct()
)
gravar_gold(bridge_movie_person, "bridge_movie_person")

bridge_movie_company = (
    s_pe.filter(F.col("tipo_entidade") == "Produtora")
    .join(mv, "id_filme")
    .join(dim_companies, F.col("nome_entidade") == F.col("nome_produtora"))
    .select("sk_movie_id", "sk_company_id").distinct()
)
gravar_gold(bridge_movie_company, "bridge_movie_company")

gold.bridge_movie_genre: 139,862 linhas
gold.bridge_movie_person: 754,068 linhas
gold.bridge_movie_company: 113,473 linhas


## Validação do modelo (integridade referencial e tipos)

In [0]:
def checar(nome_check, ok):
    print(("Foi" if ok else "Não foi"), nome_check)
    assert ok, f"Falhou: {nome_check}"


def pk_unica(tabela, pk):
    t = spark.table(f"gold.{tabela}")
    return t.count() == t.select(pk).distinct().count()


def fk_ok(tabela, fk, dim, pk):
    chaves_dim = spark.table(f"gold.{dim}").select(F.col(pk).alias("_pk_dim")).distinct()
    orfas = spark.table(f"gold.{tabela}").join(chaves_dim, F.col(fk) == F.col("_pk_dim"), "left_anti").count()
    return orfas == 0


checar("PK única: dim_movies",     pk_unica("dim_movies", "sk_movie_id"))
checar("PK única: dim_genres",     pk_unica("dim_genres", "sk_genre_id"))
checar("PK única: dim_people",     pk_unica("dim_people", "sk_person_id"))
checar("PK única: dim_companies",  pk_unica("dim_companies", "sk_company_id"))
checar("PK única: dim_reviews",    pk_unica("dim_reviews", "sk_review_id"))
checar("Grão: fact 1 linha por sk_movie_id", pk_unica("fact_movies_performance", "sk_movie_id"))
checar("Grão: dim_reviews 1 linha por filme", pk_unica("dim_reviews", "sk_movie_id"))
checar("FK fact -> dim_movies",              fk_ok("fact_movies_performance", "sk_movie_id", "dim_movies", "sk_movie_id"))
checar("FK dim_reviews -> dim_movies",       fk_ok("dim_reviews", "sk_movie_id", "dim_movies", "sk_movie_id"))
checar("FK bridge_genre -> dims",   fk_ok("bridge_movie_genre", "sk_movie_id", "dim_movies", "sk_movie_id") and fk_ok("bridge_movie_genre", "sk_genre_id", "dim_genres", "sk_genre_id"))
checar("FK bridge_person -> dims",  fk_ok("bridge_movie_person", "sk_movie_id", "dim_movies", "sk_movie_id") and fk_ok("bridge_movie_person", "sk_person_id", "dim_people", "sk_person_id"))
checar("FK bridge_company -> dims", fk_ok("bridge_movie_company", "sk_movie_id", "dim_movies", "sk_movie_id") and fk_ok("bridge_movie_company", "sk_company_id", "dim_companies", "sk_company_id"))
checar("dim_people só com Ator/Diretor/Roteirista",
       spark.table("gold.dim_people").filter(~F.col("tipo_pessoa").isin("Ator", "Diretor", "Roteirista")).count() == 0)

spark.table("gold.fact_movies_performance").printSchema()

Foi PK única: dim_movies
Foi PK única: dim_genres
Foi PK única: dim_people
Foi PK única: dim_companies
Foi PK única: dim_reviews
Foi Grão: fact 1 linha por sk_movie_id
Foi Grão: dim_reviews 1 linha por filme
Foi FK fact -> dim_movies
Foi FK dim_reviews -> dim_movies
Foi FK bridge_genre -> dims
Foi FK bridge_person -> dims
Foi FK bridge_company -> dims
Foi dim_people só com Ator/Diretor/Roteirista
root
 |-- sk_movie_id: long (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)



# 4. Desafio de Analytics — Perguntas de Negócio

### 1. Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
display(
    spark.table("gold.fact_movies_performance")
    .agg(F.sum("receita_brl").cast("decimal(24,2)").alias("receita_total_brl"),
         F.sum("receita_usd").cast("decimal(24,2)").alias("receita_total_usd"),
         F.count("receita_brl").alias("qtd_filmes_com_receita"))
)

receita_total_brl,receita_total_usd,qtd_filmes_com_receita
810384063023.60,162307288955.00,3266


### 2. Quais são os 5 filmes com maior popularidade?

In [0]:
display(
    spark.table("gold.fact_movies_performance").join(spark.table("gold.dim_movies"), "sk_movie_id")
    .filter(F.col("popularidade").isNotNull())
    .select("titulo", "popularidade")
    .orderBy(F.col("popularidade").desc())
    .limit(5)
)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
Retribution,1547.22


### 3. Quantos filmes cada gênero possui? (do maior para o menor)

In [0]:
display(
    spark.table("gold.bridge_movie_genre")
    .join(spark.table("gold.dim_genres"), "sk_genre_id")
    .groupBy("nome_genero")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc(), "nome_genero")
)

nome_genero,qtd_filmes
Drama,32127
Documentary,18928
Comedy,18537
Thriller,10242
Horror,9674
Romance,7619
Action,6039
Crime,4723
Animation,4454
TV Movie,4066


### 4. Top 10 filmes por receita — título, receita (US$ e R$) e posição (RANK())

In [0]:
spark.table("gold.fact_movies_performance").join(spark.table("gold.dim_movies"), "sk_movie_id") \
     .filter(F.col("receita_usd").isNotNull()).createOrReplaceTempView("vw_filmes_receita")

display(spark.sql('''
    SELECT titulo, receita_usd, receita_brl, posicao_ranking
    FROM (
        SELECT titulo, receita_usd, receita_brl,
               RANK() OVER (ORDER BY receita_usd DESC) AS posicao_ranking
        FROM vw_filmes_receita
    )
    WHERE posicao_ranking <= 10
    ORDER BY posicao_ranking, titulo
'''))

titulo,receita_usd,receita_brl,posicao_ranking
Avengers: Endgame,2800000000.00,13980120000.00,1
Avatar: The Way of Water,2320250281.00,11584777628.00,2
AVENGERS: INFINITY WAR,2052415039.00,10247503048.22,3
spider-man: no way home,1921847111.00,9595590440.51,4
The Lion King,1663075401.00,8303569169.65,5
Top Gun: Maverick,1488732821.00,7433094101.97,6
Barbie,1428545028.00,7132582470.30,7
The Super Mario Bros. Movie,1355725263.00,6769000665.63,8
Black Panther,1349926083.00,6740045939.81,9
Star Wars: The Last Jedi,1332698830.00,6654031988.31,10


### Recorte temporal (perguntas 5 e 6)
Data-limite superior = **data de lançamento mais recente da base**, considerando só filmes **lançados** e **ignorando datas futuras** (`> current_date()`).

In [0]:
DATA_LIMITE = (
    spark.table("gold.dim_movies")
    .filter((F.col("status_filme") == "Lançado") & F.col("data_lancamento").isNotNull() & (F.col("data_lancamento") <= F.current_date()))
    .agg(F.max("data_lancamento")).first()[0]
)
print("Data-limite superior:", DATA_LIMITE)

# Filmes lançados dentro da janela [limite - N anos, limite]
def filmes_na_janela(anos):
    return (spark.table("gold.dim_movies")
            .filter((F.col("status_filme") == "Lançado")
                    & (F.col("data_lancamento") <= F.lit(DATA_LIMITE))
                    & (F.col("data_lancamento") >= F.add_months(F.lit(DATA_LIMITE), -12 * anos)))
            .select("sk_movie_id"))

Data-limite superior: 2026-02-19


### 5. Qual ator teve a maior quantidade de participações em filmes lançados nos últimos 2 anos?

In [0]:
atores_2a = (
    filmes_na_janela(2)
    .join(spark.table("gold.bridge_movie_person"), "sk_movie_id")
    .join(spark.table("gold.dim_people").filter(F.col("tipo_pessoa") == "Ator"), "sk_person_id")
    .groupBy("nome_pessoa")
    .agg(F.countDistinct("sk_movie_id").alias("qtd_participacoes"))
    .orderBy(F.col("qtd_participacoes").desc(), "nome_pessoa")
)
display(atores_2a.limit(1)) # a 1ª linha é a resposta (demais linhas permitem ver empates)

nome_pessoa,qtd_participacoes
Kevin Hart,64


### 6. Qual produtora teve o maior lucro nos últimos 5 anos?

In [0]:
produtoras_5a = (
    filmes_na_janela(5)
    .join(spark.table("gold.fact_movies_performance").filter(F.col("lucro_usd").isNotNull()), "sk_movie_id")
    .join(spark.table("gold.bridge_movie_company"), "sk_movie_id")
    .join(spark.table("gold.dim_companies"), "sk_company_id")
    .groupBy("nome_produtora")
    .agg(F.sum("lucro_usd").cast("decimal(24,2)").alias("lucro_total_usd"),
         F.sum("lucro_brl").cast("decimal(24,2)").alias("lucro_total_brl"),
         F.countDistinct("sk_movie_id").alias("qtd_filmes_com_lucro_calculado"))
    .orderBy(F.col("lucro_total_usd").desc())
)
display(produtoras_5a.limit(1)) # a 1ª linha é a resposta

nome_produtora,lucro_total_usd,lucro_total_brl,qtd_filmes_com_lucro_calculado
Universal Pictures,5772329679.00,28820664854.27,24
